In [1]:
# ── Cell 1: Imports & global config ──────────────────────────────
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
# Reproducibility — same random numbers every run
torch.manual_seed(1337)

In [4]:
# Hyperparameters
# We WILL tune these later; keeping them in one place = easy to change.
config = {
    "batch_size":   32,     # how many sequences we process in parallel
    "block_size":   64,     # max context length (chars the model "sees")
    "n_embd":       128,    # embedding dimension (model width)
    "n_head":       4,      # number of attention heads
    "n_layer":      4,      # number of transformer blocks (depth)
    "dropout":      0.1,    # regularization
    "learning_rate": 3e-4,
    "max_iters":    3000,   # training steps
    "eval_interval": 300,   # how often we check val loss
    "eval_iters":   200,    # batches to average for a stable loss estimate
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
}

print("Running on:", config["device"])

Running on: cpu


In [5]:
# ── Cell 2: Load data & build char-level tokenizer ───────────────
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print("First 100 chars:\n", text[:100])

Total characters: 1115394
First 100 chars:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [6]:
# The "vocabulary" = every unique character in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("\nVocab size:", vocab_size)
print("Characters:", "".join(chars))


Vocab size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [7]:
# Tokenizer: map each char <-> an integer.
# This is the SIMPLEST possible tokenizer — one char = one token.
stoi = {ch: i for i, ch in enumerate(chars)}   # string-to-int
itos = {i: ch for i, ch in enumerate(chars)}   # int-to-string

In [10]:
encode = lambda s: [stoi[c] for c in s]        # "hi" -> [46, 47]
decode = lambda l: "".join([itos[i] for i in l])  # [46,47] -> "hi"

# Quick sanity check
print("\nEncoded 'hii there':", encode("hii there"))
print("Decoded back:", decode(encode("hii there")))


Encoded 'hii there': [46, 47, 47, 1, 58, 46, 43, 56, 43]
Decoded back: hii there


In [11]:
# ── Cell 3: Encode full dataset & split train/val ────────────────
# Turn the ENTIRE text into one long tensor of integers.
data = torch.tensor(encode(text), dtype=torch.long)
print("Data shape:", data.shape, "| dtype:", data.dtype)
print("First 50 tokens:", data[:50])

Data shape: torch.Size([1115394]) | dtype: torch.int64
First 50 tokens: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [12]:
# Split: 90% train, 10% validation.
# Val set = text the model NEVER trains on, so we can honestly
# measure whether it's learning patterns vs. just memorizing.
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

In [ ]:
# decode(train_data[:100].tolist())
# decode(val_data[:100].tolist())

'?\n\nGREMIO:\nGood morrow, neighbour Baptista.\n\nBAPTISTA:\nGood morrow, neighbour Gremio.\nGod save you, '

In [20]:
print("\nTrain tokens:", len(train_data))
print("Val tokens:  ", len(val_data))


Train tokens: 1003854
Val tokens:   111540


In [21]:
# ── Cell 4: Batch generator ──────────────────────────────────────
def get_batch(split):
    # Pick the right ribbon of data
    data = train_data if split == "train" else val_data

    # Random starting positions — one per sequence in the batch.
    # We need room for block_size chars + 1 (the target), so we stop
    # ix at len(data) - block_size.
    ix = torch.randint(len(data) - config["block_size"], (config["batch_size"],))

    # x = input windows;  y = same windows shifted right by 1 (the "next char")
    x = torch.stack([data[i     : i + config["block_size"]]     for i in ix])
    y = torch.stack([data[i + 1 : i + config["block_size"] + 1] for i in ix])

    x, y = x.to(config["device"]), y.to(config["device"])
    return x, y

In [22]:
# Sanity check
xb, yb = get_batch("train")
print("Inputs  x shape:", xb.shape)   # (batch_size, block_size)
print("Targets y shape:", yb.shape)   # (batch_size, block_size)
print("\nExample — first sequence in batch:")
print("x[0]:", xb[0].tolist())
print("y[0]:", yb[0].tolist())

Inputs  x shape: torch.Size([32, 64])
Targets y shape: torch.Size([32, 64])

Example — first sequence in batch:
x[0]: [50, 43, 8, 1, 32, 46, 47, 52, 49, 1, 61, 47, 58, 46, 1, 58, 46, 63, 57, 43, 50, 44, 0, 20, 53, 61, 1, 51, 53, 56, 43, 1, 59, 52, 44, 53, 56, 58, 59, 52, 39, 58, 43, 1, 58, 46, 39, 52, 1, 39, 50, 50, 1, 50, 47, 60, 47, 52, 45, 1, 61, 53, 51, 43]
y[0]: [43, 8, 1, 32, 46, 47, 52, 49, 1, 61, 47, 58, 46, 1, 58, 46, 63, 57, 43, 50, 44, 0, 20, 53, 61, 1, 51, 53, 56, 43, 1, 59, 52, 44, 53, 56, 58, 59, 52, 39, 58, 43, 1, 58, 46, 39, 52, 1, 39, 50, 50, 1, 50, 47, 60, 47, 52, 45, 1, 61, 53, 51, 43, 52]
